In [35]:
!pip install nltk scikit-learn pandas

import pandas as pd
import nltk
import string
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

nltk.download('stopwords')
nltk.download('wordnet')

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [32]:
from google.colab import files
uploaded = files.upload()



Saving Resume.csv to Resume (1).csv


In [36]:
df = pd.read_csv("Resume.csv")

print("Dataset Loaded Successfully!")
print(df.head())


Dataset Loaded Successfully!
         ID                                         Resume_str  \
0  16852973           HR ADMINISTRATOR/MARKETING ASSOCIATE\...   
1  22323967           HR SPECIALIST, US HR OPERATIONS      ...   
2  33176873           HR DIRECTOR       Summary      Over 2...   
3  27018550           HR SPECIALIST       Summary    Dedica...   
4  17812897           HR MANAGER         Skill Highlights  ...   

                                         Resume_html Category  
0  <div class="fontsize fontface vmargins hmargin...       HR  
1  <div class="fontsize fontface vmargins hmargin...       HR  
2  <div class="fontsize fontface vmargins hmargin...       HR  
3  <div class="fontsize fontface vmargins hmargin...       HR  
4  <div class="fontsize fontface vmargins hmargin...       HR  


In [37]:
print("Columns:", df.columns)

Columns: Index(['ID', 'Resume_str', 'Resume_html', 'Category'], dtype='object')


In [38]:
df = df.dropna(subset=['Resume_str'])
df['Resume_str'] = df['Resume_str'].astype(str)


In [39]:
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def clean_text(text):
    text = text.lower()
    text = "".join([char for char in text if char not in string.punctuation])
    words = text.split()
    words = [lemmatizer.lemmatize(word) for word in words if word not in stop_words]
    return " ".join(words)

df['cleaned_resume'] = df['Resume_str'].apply(clean_text)

In [53]:
jobs = {
    "Data Scientist": "Python machine learning data analysis statistics pandas numpy",
    "Backend Developer": "Java spring boot mysql api backend development",
    "Frontend Developer": "HTML CSS JavaScript React frontend UI design",
     "Machine Learning Engineer":"machine learning deep learning TensorFlow PyTorch model training deployment feature engineering GPU",
}

In [49]:
skills_list = [
    # Programming Languages
    "python", "java", "javascript", "c++", "c#", "swift", "kotlin", "r", "scala", "go",
    # Web
    "html", "css", "react", "angular", "vue", "node", "bootstrap", "typescript",
    # Data & ML
    "pandas", "numpy", "scikit-learn", "tensorflow", "pytorch", "keras", "matplotlib",
    "machine learning", "deep learning", "nlp", "computer vision",
    # Cloud & DevOps
    "aws", "azure", "gcp", "docker", "kubernetes", "jenkins", "linux", "git", "terraform",
    # Analytics
    "tableau", "power bi", "excel", "statistics",
    # Mobile
    "android", "ios",
    # Security
    "penetration testing", "network security", "encryption",
]
def extract_skills(text):
    return list(set([skill for skill in skills_list if skill in text]))

df['skills'] = df['cleaned_resume'].apply(extract_skills)


In [44]:
def match_resume_to_jobs(resume_text, resume_skills):
    results = []

    for job, desc in jobs.items():
        corpus = [resume_text, clean_text(desc)]

        vectorizer = TfidfVectorizer(ngram_range=(1,2))
        tfidf_matrix = vectorizer.fit_transform(corpus)

        similarity = cosine_similarity(tfidf_matrix[0:1], tfidf_matrix[1:2])[0][0]

        job_skills = extract_skills(desc)
        skill_match = len(set(resume_skills) & set(job_skills)) / (len(job_skills) + 1e-5)

        final_score = (0.7 * similarity) + (0.3 * skill_match)

        results.append((job, round(final_score * 100, 2)))

    return sorted(results, key=lambda x: x[1], reverse=True)

In [45]:
def skill_gap(resume_skills, job_desc):
    job_skills = extract_skills(job_desc)
    return list(set(job_skills) - set(resume_skills))

In [58]:
for i in range(min(10, len(df))):
    print("\n" + "."*40)

    print(f"Candidate ID: {df.loc[i, 'ID']}")
    print(f"Skills: {df.loc[i, 'skills']}")


    matches = match_resume_to_jobs(
        df.loc[i, 'cleaned_resume'],
        df.loc[i, 'skills']
    )

    print("\nJob Match Scores:")
    for job, score in matches:
        print(f"  {job}: {score}%")

    best_job = matches[0][0]
    missing_skills = skill_gap(df.loc[i, 'skills'], jobs[best_job])

    print(f"\nRecommended Role: {best_job}")
    print(f" Missing Skills: {missing_skills}")



........................................
Candidate ID: 16852973
Skills: ['swift', 'go', 'r']

Job Match Scores:
  Frontend Developer: 30.41%
  Backend Developer: 30.2%
  Machine Learning Engineer: 10.73%
  Data Scientist: 7.78%

Recommended Role: Frontend Developer
 Missing Skills: []

........................................
Candidate ID: 22323967
Skills: ['r', 'go', 'git']

Job Match Scores:
  Backend Developer: 30.42%
  Frontend Developer: 30.42%
  Machine Learning Engineer: 10.46%
  Data Scientist: 6.19%

Recommended Role: Backend Developer
 Missing Skills: []

........................................
Candidate ID: 33176873
Skills: ['r', 'excel', 'go']

Job Match Scores:
  Backend Developer: 31.75%
  Frontend Developer: 30.0%
  Machine Learning Engineer: 10.5%
  Data Scientist: 6.0%

Recommended Role: Backend Developer
 Missing Skills: []

........................................
Candidate ID: 27018550
Skills: ['r', 'excel']

Job Match Scores:
  Backend Developer: 30.0%
  Frontend